# Traefik Routing

Traefik is the reverse proxy and SSL terminator. Coolify deploys it automatically.
This notebook covers how routes work, how to add one, and how to debug when traffic is not reaching your container.

## How Traefik Works

Traefik is a **dynamic** reverse proxy. It discovers routes by watching Docker labels on running containers.
You do not edit a config file to add a route; you add labels to the container.

```
Internet
   |
   v
Traefik (port 80/443)
   |     |
   v     v
 app1  app2  (containers, matched by Host header)
```

Let's Encrypt handles SSL. Traefik requests and renews certs automatically.

## Adding a Route via Docker Labels

In your `docker-compose.yml` (or Coolify's environment variables):

```yaml
services:
  my-app:
    image: my-image
    labels:
      - "traefik.enable=true"
      - "traefik.http.routers.my-app.rule=Host(`myapp.yourdomain.com`)"
      - "traefik.http.routers.my-app.entrypoints=websecure"
      - "traefik.http.routers.my-app.tls=true"
      - "traefik.http.routers.my-app.tls.certresolver=letsencrypt"
      - "traefik.http.services.my-app.loadbalancer.server.port=3000"
    networks:
      - traefik-net

networks:
  traefik-net:
    external: true
    name: coolify  # Coolify's network name
```

**The container must be on the same Docker network as Traefik.** In Coolify this is the `coolify` network.

## Via Coolify

When you set a domain in Coolify's application settings, Coolify injects all the required Traefik labels automatically. You do not need to write them yourself.

1. Application > Settings > Domains
2. Enter `https://myapp.yourdomain.com`
3. Coolify handles: labels, cert request, HTTP to HTTPS redirect

Use manual labels only when deploying outside Coolify.

## Middleware

Traefik middleware runs before the request reaches your service.

### HTTP to HTTPS redirect

```yaml
labels:
  - "traefik.http.middlewares.redirect-https.redirectscheme.scheme=https"
  - "traefik.http.routers.my-app-http.rule=Host(`myapp.yourdomain.com`)"
  - "traefik.http.routers.my-app-http.entrypoints=web"
  - "traefik.http.routers.my-app-http.middlewares=redirect-https"
```

### Basic auth

```bash
# Generate htpasswd
htpasswd -nb admin yourpassword
# Output: admin:$apr1$...
```

```yaml
labels:
  - "traefik.http.middlewares.auth.basicauth.users=admin:$$apr1$$..."
  # Note: $ must be escaped as $$ in yaml
  - "traefik.http.routers.my-app.middlewares=auth"
```

### Path prefix stripping

```yaml
labels:
  - "traefik.http.routers.my-app.rule=Host(`yourdomain.com`) && PathPrefix(`/app`)"
  - "traefik.http.middlewares.strip-app.stripprefix.prefixes=/app"
  - "traefik.http.routers.my-app.middlewares=strip-app"
```

## Troubleshooting

### Route not working

```bash
# 1. Is Traefik running?
docker ps | grep traefik

# 2. Is the container on the right network?
docker inspect MY_CONTAINER | jq '.[0].NetworkSettings.Networks | keys'
# Should include "coolify"

# 3. Does Traefik see the route?
# Open: http://YOUR_IP:8080 (Traefik dashboard)
# Or check logs:
docker logs traefik --tail 100 2>&1 | grep -i 'my-app\|error'

# 4. Is DNS resolving to the server?
dig myapp.yourdomain.com +short

# 5. Is SSL cert provisioned?
curl -I https://myapp.yourdomain.com
```

### SSL cert not provisioning

```bash
# Check Let's Encrypt logs
docker logs traefik 2>&1 | grep -i 'acme\|cert\|challenge'

# Common causes:
# - Port 80 not open in UFW (HTTP-01 challenge needs port 80)
# - DNS not yet propagated
# - Let's Encrypt rate limit hit (20 certs per domain per week)
```

## Traefik Route Skill

The `traefik-route` tool at `~/dev/mad-house/traefik-route/` (also in the madebymadhouse/traefik-route repo) automates adding and removing routes.

```bash
# Add a route
traefik-route add --domain myapp.yourdomain.com --container MY_CONTAINER --port 3000

# List all routes
traefik-route list

# Remove a route
traefik-route remove --domain myapp.yourdomain.com
```

See the repo README for full usage.